Importando as bibliotecas

In [22]:
import pandas as pd
import numpy as np
import uuid6
import psycopg2
import os
import io
from dotenv import load_dotenv

Extraindo o dataframe

In [23]:
df = pd.read_csv("../data/Vendas_varejo.csv")

Realizando uma análise panorâmica sobre o df

In [24]:
#Verificando uma amostra
df.sample(15)

,id_venda,data_venda,nome_cliente,cidade,produto,categoria,quantidade,preco_unitario
67,68,2023-04-12,Ana Silva,Salvador,Sandália,Calçados,3,"89,9"
184,185,2023-01-13,Thiago Barbosa,Fortaleza,Carteira,Acessórios,3,59.9
92,93,2023-06-17,Juliana Mendes,Curitiba,Tênis Casual,Calçados,4,199.9
141,142,2024/05/04,Diego Martins,Belo Horizonte,Boné,Roupas,1,R$ 39.9
265,266,14/11/2024,Rafael Alves,Manaus,Cinto,acessórios,3,44.9
291,292,01/05/2024,PEDRO SANTOS,Brasília,Sandália,Calçados,0,R$ 89.9
56,57,2024-12-26,NaN,rio de janeiro,Carteira,acessórios,3,59.9
240,241,2024-05-23,Lucas Rocha,Belo Horizonte,Boné,roupas,4,39.9
80,81,2023-08-11,Pedro Santos,Porto Alegre,Tênis Casual,calçados,1,199.9
147,148,2023-11-30,Maria Oliveira,Salvador,Chinelo,Calçados,2,49.9


In [25]:
# Verificando informações gerais
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 315 entries, 0 to 314
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id_venda        315 non-null    int64 
 1   data_venda      304 non-null    object
 2   nome_cliente    298 non-null    object
 3   cidade          315 non-null    object
 4   produto         315 non-null    object
 5   categoria       315 non-null    object
 6   quantidade      315 non-null    int64 
 7   preco_unitario  302 non-null    object
dtypes: int64(2), object(6)
memory usage: 19.8+ KB


In [26]:
# Verificando informações estatísticas
df.describe()

,id_venda,quantidade
count,315.000000,315.000000
mean,158.000000,2.726984
std,91.076891,1.533616
min,1.000000,-1.000000
25%,79.500000,2.000000
50%,158.000000,3.000000
75%,236.500000,4.000000
max,315.000000,5.000000


In [27]:
# Verificando linhas duplicadas
int(df.duplicated(subset=df.columns.drop("id_venda")).sum())

15

Limpeza do dataset

In [28]:
# Criando uma cópia para evitar futuros erros
df = df.copy()

In [29]:
# Retirando as linhas repetidas
df = df.drop_duplicates(subset=df.columns.drop("id_venda"))

In [30]:
# Retirando as linhas dessas duas colunas que tenham NaN
df = df.dropna(subset=["data_venda","nome_cliente"])

In [31]:
# Ajustando formatação do preço e da data
df["preco_unitario"] = (df["preco_unitario"]
    .str.replace("R$", "", regex=False)
    .str.strip()
    .str.replace(",", ".")
    .astype(float))

df["data_venda"] = pd.to_datetime(df["data_venda"].replace("[/]","-", regex=True), dayfirst=True, format='mixed')


In [32]:
# Colocando essas colunas em minúsculo pois é mais fácil de trabalhar
cols = ["nome_cliente", "cidade", "produto", "categoria"]

df[cols] = df[cols].apply(lambda x: x.str.lower()).astype(str)


In [33]:
# Preenchendo os NaN do preco_unitario com a média
def get_price(x:str):
    return float(df[df["produto"] == x]["preco_unitario"].dropna().mean())

produtos = df["produto"].drop_duplicates().to_list()

preco_produto = {}

for i in produtos:
    preco_produto[i] = get_price(i)

df["preco_unitario"] = df["preco_unitario"].fillna(df["produto"].map(preco_produto))

In [34]:
# Organizando pela data da venda
df = df.sort_values(by="data_venda")

In [35]:
# Retirando linhas com quantidade = 0 ou negativa
df = df.drop(df[df["quantidade"] < 1].index)

In [36]:
# Adicionando uma ID UUIDv7
unic_ids = []

for i in range(len(df)):
    unic_ids.append(str(uuid6.uuid7()))

df["id_venda"] = unic_ids

In [37]:
df

,id_venda,data_venda,nome_cliente,cidade,produto,categoria,quantidade,preco_unitario
139,019db520-96ab-757f-a43e-6a54cf7b382e,2023-01-02,diego martins,rio de janeiro,calça jeans,roupas,2,129.9
277,019db520-96ac-747e-aa09-e5cb9260d7b5,2023-01-07,thiago barbosa,são paulo,calça jeans,roupas,4,129.9
184,019db520-96ad-72bc-86da-119febc9576f,2023-01-13,thiago barbosa,fortaleza,carteira,acessórios,3,59.9
70,019db520-96ae-7ef2-829c-9e9ff5e45306,2023-01-20,larissa nunes,rio de janeiro,camiseta básica,roupas,3,49.9
24,019db520-96af-75cd-a59a-bcca08195bb8,2023-01-20,lucas rocha,recife,cinto,acessórios,1,44.9
...,...,...,...,...,...,...,...,...
75,019db520-97a8-73f8-bdd2-32d3e11b2736,2024-12-20,patricia gomes,são paulo,camiseta básica,roupas,2,49.9
272,019db520-97a9-7309-bb47-fe7100e064d7,2024-12-21,rafael alves,manaus,tênis casual,calçados,3,199.9
203,019db520-97aa-71a4-82f0-9249ef4c1442,2024-12-22,maria oliveira,porto alegre,calça jeans,roupas,5,129.9
81,019db520-97ab-7ce2-bbd3-1eaa7d7f87f8,2024-12-24,pedro santos,porto alegre,camiseta básica,roupas,3,49.9


In [38]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 258 entries, 139 to 248
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   id_venda        258 non-null    object        
 1   data_venda      258 non-null    datetime64[ns]
 2   nome_cliente    258 non-null    object        
 3   cidade          258 non-null    object        
 4   produto         258 non-null    object        
 5   categoria       258 non-null    object        
 6   quantidade      258 non-null    int64         
 7   preco_unitario  258 non-null    float64       
dtypes: datetime64[ns](1), float64(1), int64(1), object(5)
memory usage: 18.1+ KB


Carregamento do dataset na camada silver do PostgreSQL

In [39]:
buffer = io.StringIO()
df.to_csv(buffer, index=False)
buffer.seek(0)

0

In [40]:
load_dotenv(dotenv_path="../../.env")

conn = psycopg2.connect(
    host=os.getenv("host"),
    dbname=os.getenv("dbname"),
    user=os.getenv("user"),
    password=os.getenv("password"),
    port=os.getenv("port")
)

In [41]:
cur = conn.cursor()

cur.execute("CREATE SCHEMA IF NOT EXISTS case_1;")

cur.execute("""
CREATE TABLE IF NOT EXISTS case_1.vendas_silver (
    id_venda VARCHAR(40) PRIMARY KEY,
    data_venda TIMESTAMP,
    nome_cliente VARCHAR(255),
    cidade VARCHAR(100),
    produto VARCHAR(255),
    categoria VARCHAR(100),
    quantidade INT,
    preco_unitario DECIMAL(10, 2)
);
""")

cur.copy_expert(sql="COPY case_1.vendas_silver FROM STDIN WITH CSV HEADER", file=buffer)

conn.commit() 
cur.close() 
conn.close()

